In [ ]:
import os
import pandas as pd
import numpy as np

LABEL_PKL = "/mnt/home/qinqiu/myproject/Combined_data/filtered_dataset.pkl"
OUT_DIR = "/mnt/home/qinqiu/thesis/data"
os.makedirs(OUT_DIR, exist_ok=True)

df_label = pd.read_pickle(LABEL_PKL)

id_col = "ids__uid"
pma_col = "target_pma_w"
nec_col = "target__abdominal_nec"
los_col = "target__los"

required_cols = [id_col, pma_col, nec_col, los_col]
missing = [c for c in required_cols if c not in df_label.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ============================================================
# 1) Clean required columns
# ============================================================
tmp = df_label[[id_col, pma_col, nec_col, los_col]].copy()

tmp[id_col] = tmp[id_col].astype(str)
tmp[pma_col] = pd.to_numeric(tmp[pma_col], errors="coerce")
tmp[nec_col] = pd.to_numeric(tmp[nec_col], errors="coerce").fillna(0).astype(int)
tmp[los_col] = pd.to_numeric(tmp[los_col], errors="coerce").fillna(0).astype(int)

n_before_drop = len(tmp)
tmp = tmp.dropna(subset=[id_col, pma_col]).copy()
n_after_drop = len(tmp)

print(f"Rows with valid UID and PMA: {n_before_drop} -> {n_after_drop}")

# adverse event = NEC or LOS
tmp["adverse_event"] = (
    (tmp[nec_col] == 1) |
    (tmp[los_col] == 1)
).astype(int)


# ============================================================
# 2) Collapse to patient-PMA level
#    If any row/window at the same PMA is positive, this PMA is positive
# ============================================================
patient_pma = (
    tmp
    .groupby([id_col, pma_col], as_index=False)[[nec_col, los_col, "adverse_event"]]
    .max()
    .sort_values([id_col, pma_col])
)

print("Patient-PMA table shape:", patient_pma.shape)


# ============================================================
# 3) Patient-level adverse event IDs
# ============================================================
patient_level = (
    patient_pma
    .groupby(id_col, as_index=False)[[nec_col, los_col, "adverse_event"]]
    .max()
)

nec_ids = sorted(patient_level.loc[patient_level[nec_col] == 1, id_col].tolist())
los_ids = sorted(patient_level.loc[patient_level[los_col] == 1, id_col].tolist())
nec_los_ids = sorted(patient_level.loc[patient_level["adverse_event"] == 1, id_col].tolist())

print("NEC patients:", len(nec_ids))
print("LOS patients:", len(los_ids))
print("NEC or LOS patients:", len(nec_los_ids))


# ============================================================
# 4) Keep PMA timeline only for adverse-event patients
# ============================================================
adverse_event_timeline = patient_pma[
    patient_pma[id_col].isin(nec_los_ids)
].copy()

# Reorder columns
adverse_event_timeline = adverse_event_timeline[
    [id_col, pma_col, nec_col, los_col, "adverse_event"]
]

print("Adverse-event patient-PMA timeline shape:", adverse_event_timeline.shape)


# ============================================================
# 5) Summarize positive and negative PMA values per patient
# ============================================================
def pma_list_to_str(values):
    values = (
        pd.Series(values)
        .dropna()
        .astype(float)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    return ";".join(f"{v:g}" for v in values)


def first_value(values):
    values = (
        pd.Series(values)
        .dropna()
        .astype(float)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    return values[0] if len(values) > 0 else np.nan


def last_negative_before_first_positive(g, event_col):
    pos_values = (
        g.loc[g[event_col] == 1, pma_col]
        .dropna()
        .astype(float)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    if len(pos_values) == 0:
        return np.nan

    first_pos = pos_values[0]

    neg_before = (
        g.loc[(g[event_col] == 0) & (g[pma_col] < first_pos), pma_col]
        .dropna()
        .astype(float)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    return neg_before[-1] if len(neg_before) > 0 else np.nan


summary_rows = []

for pid, g in adverse_event_timeline.groupby(id_col):
    g = g.sort_values(pma_col)

    row = {
        id_col: pid,

        "nec_patient": int(g[nec_col].max()),
        "los_patient": int(g[los_col].max()),
        "adverse_event_patient": int(g["adverse_event"].max()),

        "n_pma_points": g[pma_col].nunique(),

        # Combined adverse event: NEC or LOS
        "adverse_positive_pma_w": pma_list_to_str(g.loc[g["adverse_event"] == 1, pma_col]),
        "adverse_negative_pma_w": pma_list_to_str(g.loc[g["adverse_event"] == 0, pma_col]),
        "first_adverse_positive_pma_w": first_value(g.loc[g["adverse_event"] == 1, pma_col]),
        "last_adverse_negative_before_positive_pma_w": last_negative_before_first_positive(g, "adverse_event"),

        # NEC-specific
        "nec_positive_pma_w": pma_list_to_str(g.loc[g[nec_col] == 1, pma_col]),
        "nec_negative_pma_w": pma_list_to_str(g.loc[g[nec_col] == 0, pma_col]),
        "first_nec_positive_pma_w": first_value(g.loc[g[nec_col] == 1, pma_col]),
        "last_nec_negative_before_positive_pma_w": last_negative_before_first_positive(g, nec_col),

        # LOS-specific
        "los_positive_pma_w": pma_list_to_str(g.loc[g[los_col] == 1, pma_col]),
        "los_negative_pma_w": pma_list_to_str(g.loc[g[los_col] == 0, pma_col]),
        "first_los_positive_pma_w": first_value(g.loc[g[los_col] == 1, pma_col]),
        "last_los_negative_before_positive_pma_w": last_negative_before_first_positive(g, los_col),
    }

    summary_rows.append(row)

adverse_event_summary = pd.DataFrame(summary_rows)

print("Adverse-event summary shape:", adverse_event_summary.shape)


# ============================================================
# 6) Save outputs
# ============================================================

# Patient ID lists
pd.Series(nec_ids, name=id_col).to_csv(
    os.path.join(OUT_DIR, "ids_nec.csv"),
    index=False
)

pd.Series(los_ids, name=id_col).to_csv(
    os.path.join(OUT_DIR, "ids_los.csv"),
    index=False
)

pd.Series(nec_los_ids, name=id_col).to_csv(
    os.path.join(OUT_DIR, "ids_nec_or_los.csv"),
    index=False
)

# Same content, but compatible with your later cleaning code name
pd.Series(nec_los_ids, name=id_col).to_csv(
    os.path.join(OUT_DIR, "sick_uids_los_or_nec.csv"),
    index=False
)

# Long-format timeline: one row per adverse patient per PMA
adverse_event_timeline.to_csv(
    os.path.join(OUT_DIR, "adverse_event_pma_timeline.csv"),
    index=False
)

# Patient-level summary
adverse_event_summary.to_csv(
    os.path.join(OUT_DIR, "adverse_event_pma_summary.csv"),
    index=False
)

print("Saved:")
print(os.path.join(OUT_DIR, "ids_nec.csv"))
print(os.path.join(OUT_DIR, "ids_los.csv"))
print(os.path.join(OUT_DIR, "ids_nec_or_los.csv"))
print(os.path.join(OUT_DIR, "sick_uids_los_or_nec.csv"))
print(os.path.join(OUT_DIR, "adverse_event_pma_timeline.csv"))
print(os.path.join(OUT_DIR, "adverse_event_pma_summary.csv"))

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd


# ============================================================
# 0) Paths
# ============================================================

MAIN_PKL = "/mnt/home/qinqiu/thesis/data/main_data__keep_all_rows_for_nec_or_los_patients.pkl"
LABEL_PKL = "/mnt/home/qinqiu/myproject/Combined_data/filtered_dataset.pkl"

OUT_DIR = "/mnt/home/qinqiu/thesis/data"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PKL = os.path.join(
    OUT_DIR,
    "main_data__keep_all_rows_for_nec_or_los_patients__with_nec_los_tags.pkl"
)


# ============================================================
# 1) Column names
# ============================================================

id_col = "ids__uid"

# Main data PMA column
main_pma_col = "target__pma_w"

# Label data PMA column
label_pma_col = "target_pma_w"

# Adverse event columns in label data
nec_col = "target__abdominal_nec"
los_col = "target__los"

# PMA is float, so round before merging to avoid tiny float mismatch
pma_round_decimals = 6


# ============================================================
# 2) Load data
# ============================================================

with open(MAIN_PKL, "rb") as f:
    df_main = pickle.load(f)

df_label = pd.read_pickle(LABEL_PKL)

if not isinstance(df_main, pd.DataFrame):
    raise ValueError("MAIN_PKL does not contain a pandas DataFrame.")

if not isinstance(df_label, pd.DataFrame):
    raise ValueError("LABEL_PKL does not contain a pandas DataFrame.")

print("Main shape:", df_main.shape)
print("Label shape:", df_label.shape)

print("\nMain PMA-related columns:")
print([c for c in df_main.columns if "pma" in c.lower()])

print("\nLabel PMA / NEC / LOS-related columns:")
print([
    c for c in df_label.columns
    if "pma" in c.lower() or "nec" in c.lower() or "los" in c.lower()
])


# ============================================================
# 3) Check required columns
# ============================================================

required_main_cols = [id_col, main_pma_col]
required_label_cols = [id_col, label_pma_col, nec_col, los_col]

missing_main = [c for c in required_main_cols if c not in df_main.columns]
missing_label = [c for c in required_label_cols if c not in df_label.columns]

if missing_main:
    raise ValueError(f"Missing columns in main data: {missing_main}")

if missing_label:
    raise ValueError(f"Missing columns in label data: {missing_label}")


# ============================================================
# 4) Clean ID / PMA / labels
# ============================================================

df_main = df_main.copy()
df_label = df_label.copy()

df_main[id_col] = df_main[id_col].astype(str)
df_label[id_col] = df_label[id_col].astype(str)

df_main[main_pma_col] = pd.to_numeric(df_main[main_pma_col], errors="coerce")
df_label[label_pma_col] = pd.to_numeric(df_label[label_pma_col], errors="coerce")

df_main = df_main.dropna(subset=[id_col, main_pma_col]).copy()
df_label = df_label.dropna(subset=[id_col, label_pma_col]).copy()

df_label[nec_col] = pd.to_numeric(df_label[nec_col], errors="coerce").fillna(0).astype(int)
df_label[los_col] = pd.to_numeric(df_label[los_col], errors="coerce").fillna(0).astype(int)

# Create unified PMA key for merging
df_main["_pma_key"] = df_main[main_pma_col].round(pma_round_decimals)
df_label["_pma_key"] = df_label[label_pma_col].round(pma_round_decimals)


# ============================================================
# 5) Build patient-PMA level label lookup
# ============================================================
# If the same patient has multiple rows at the same PMA,
# use max to preserve any positive NEC/LOS tag.

label_lookup = (
    df_label
    .groupby([id_col, "_pma_key"], as_index=False)[[nec_col, los_col]]
    .max()
)

label_lookup["target__adverse_event"] = (
    (label_lookup[nec_col] == 1) |
    (label_lookup[los_col] == 1)
).astype(int)

print("\nLabel lookup shape:", label_lookup.shape)
print("Positive NEC patient-PMA rows:", int(label_lookup[nec_col].sum()))
print("Positive LOS patient-PMA rows:", int(label_lookup[los_col].sum()))
print("Positive adverse patient-PMA rows:", int(label_lookup["target__adverse_event"].sum()))

print("NEC patients in label lookup:",
      label_lookup.loc[label_lookup[nec_col] == 1, id_col].nunique())

print("LOS patients in label lookup:",
      label_lookup.loc[label_lookup[los_col] == 1, id_col].nunique())

print("Adverse patients in label lookup:",
      label_lookup.loc[label_lookup["target__adverse_event"] == 1, id_col].nunique())


# ============================================================
# 6) Merge NEC/LOS labels into main data
# ============================================================

# Remove old tag columns if they already exist
for c in [nec_col, los_col, "target__adverse_event"]:
    if c in df_main.columns:
        df_main = df_main.drop(columns=[c])

df_tagged = df_main.merge(
    label_lookup,
    on=[id_col, "_pma_key"],
    how="left"
)

# Rows without matched label are treated as negative
df_tagged[nec_col] = df_tagged[nec_col].fillna(0).astype(int)
df_tagged[los_col] = df_tagged[los_col].fillna(0).astype(int)

df_tagged["target__adverse_event"] = (
    (df_tagged[nec_col] == 1) |
    (df_tagged[los_col] == 1)
).astype(int)


# ============================================================
# 7) Check merged result
# ============================================================

print("\nTagged main shape:", df_tagged.shape)

print("Tagged NEC positive rows:", int(df_tagged[nec_col].sum()))
print("Tagged LOS positive rows:", int(df_tagged[los_col].sum()))
print("Tagged adverse positive rows:", int(df_tagged["target__adverse_event"].sum()))

print("Tagged NEC positive patients:",
      df_tagged.loc[df_tagged[nec_col] == 1, id_col].nunique())

print("Tagged LOS positive patients:",
      df_tagged.loc[df_tagged[los_col] == 1, id_col].nunique())

print("Tagged adverse positive patients:",
      df_tagged.loc[df_tagged["target__adverse_event"] == 1, id_col].nunique())

print("\nTop adverse patients by positive rows:")
print(
    df_tagged.loc[df_tagged["target__adverse_event"] == 1]
    .groupby(id_col)
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print("\nNumber of unique PMA values per patient in tagged main data:")
print(
    df_tagged
    .groupby(id_col)[main_pma_col]
    .nunique()
    .describe()
)


# ============================================================
# 8) Save tagged main data
# ============================================================

df_tagged = df_tagged.drop(columns=["_pma_key"])

with open(OUT_PKL, "wb") as f:
    pickle.dump(df_tagged, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\nSaved tagged pkl:")
print(OUT_PKL)
print("Final tagged shape:", df_tagged.shape)

In [ ]:
import pickle
import numpy as np
import pandas as pd
from typing import List, Optional


def superwindow_crop_pkl_timebased(
    input_pkl_path: str,
    output_pkl_path: str,
    id_col: str = "ids__uid",
    timestamp_col: str = "timestamp",

    # Frame / hop definition
    frame_len_minutes: int = 10,
    frame_hop_minutes: int = 5,

    # Superwindow definition
    superwindow_minutes: int = 180,
    super_stride_minutes: int = 60,

    # QC: per-feature minimum valid frames within the superwindow
    min_valid_frames_per_feature: int = 28,

    # Timestamp alignment
    align_timestamps: bool = True,
    align_method: str = "round",  # "round" or "floor"

    # Target aggregation
    target_col: str = "target__pma_w",

    # Feature selection
    feature_prefix: str = "feats__",

    # Static columns
    static_cols: Optional[List[str]] = None,

    # Adverse event tag columns
    nec_col: str = "target__abdominal_nec",
    los_col: str = "target__los",

    # Whether QC should ignore static columns
    qc_ignore_static: bool = True,

    # Print warnings for patients with missing/multiple static values
    verbose_static_warnings: bool = True,
):
    """
    Time-based superwindow cropping per patient using timestamps.

    Main outputs:
    - X_by_patient[pid]: shape = (n_superwindows, frames_per_super, n_features)
    - y_by_patient[pid]: PMA mean for each 3h superwindow
    - tag_by_patient[pid]: NEC / LOS / adverse labels for each 3h superwindow
    - meta_by_patient[pid]: metadata for each 3h superwindow, including adverse labels

    Adverse event logic:
    - For each 3h superwindow:
        target__abdominal_nec_3h = max(target__abdominal_nec within this window)
        target__los_3h = max(target__los within this window)
        adverse_event_3h = max(target__abdominal_nec_3h, target__los_3h)

    Important:
    - If any frame inside the 3h window has NEC=1 or LOS=1,
      the whole 3h window is labeled as adverse_event_3h = 1.
    """

    if static_cols is None:
        static_cols = ["feats__ga_w", "feats__sex", "feats__bw"]

    adverse_tag_cols = [nec_col, los_col]

    # -------------------------
    # Load data
    # -------------------------
    with open(input_pkl_path, "rb") as f:
        df_all = pickle.load(f)

    if not isinstance(df_all, pd.DataFrame):
        raise ValueError("Expected input pkl to contain a pandas DataFrame.")

    df_all = df_all.copy()

    required_cols = [id_col, timestamp_col, target_col, nec_col, los_col]
    missing = [c for c in required_cols if c not in df_all.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df_all[id_col] = df_all[id_col].astype(str)
    df_all[timestamp_col] = pd.to_datetime(df_all[timestamp_col], errors="coerce")

    if df_all[timestamp_col].isna().all():
        raise ValueError(f"All values in {timestamp_col} are NaT after parsing.")

    df_all[target_col] = pd.to_numeric(df_all[target_col], errors="coerce")

    for c in adverse_tag_cols:
        df_all[c] = pd.to_numeric(df_all[c], errors="coerce").fillna(0).astype(int)

    df_all["target__adverse_event"] = (
        (df_all[nec_col] == 1) |
        (df_all[los_col] == 1)
    ).astype(int)

    # -------------------------
    # Select features by prefix
    # -------------------------
    feature_cols = [
        c for c in df_all.columns
        if c.startswith(feature_prefix) and c != "feats__pna_days"
    ]

    if len(feature_cols) == 0:
        raise ValueError(f"No feature columns found with prefix '{feature_prefix}'.")

    for c in static_cols:
        if c in df_all.columns and c not in feature_cols:
            feature_cols.append(c)

    df_all = df_all.sort_values([id_col, timestamp_col], kind="stable").reset_index(drop=True)

    # -------------------------
    # Convert minutes -> slot counts
    # -------------------------
    frames_per_super = int((superwindow_minutes - frame_len_minutes) / frame_hop_minutes) + 1

    if frames_per_super <= 0:
        raise ValueError("superwindow_minutes too small compared to frame_len_minutes.")

    if min_valid_frames_per_feature > frames_per_super:
        raise ValueError("min_valid_frames_per_feature cannot exceed frames_per_super.")

    hop = pd.Timedelta(minutes=frame_hop_minutes)
    super_len = pd.Timedelta(minutes=superwindow_minutes)
    super_stride = pd.Timedelta(minutes=super_stride_minutes)

    # -------------------------
    # Helper: align timestamps to hop grid
    # -------------------------
    def align_to_hop_grid(ts: pd.Series) -> pd.Series:
        mins = (ts.astype("int64") // (60 * 10**9)).astype(np.int64)
        h = frame_hop_minutes

        if align_method == "round":
            aligned = ((mins + h // 2) // h) * h
        elif align_method == "floor":
            aligned = (mins // h) * h
        else:
            raise ValueError("align_method must be 'round' or 'floor'.")

        return pd.to_datetime(aligned, unit="m")

    def pma_list_to_str(values):
        values = (
            pd.Series(values)
            .dropna()
            .astype(float)
            .drop_duplicates()
            .sort_values()
            .tolist()
        )
        return ";".join(f"{v:g}" for v in values)

    def safe_nanmean(arr):
        arr = np.asarray(arr, dtype=float)
        if np.isfinite(arr).any():
            return float(np.nanmean(arr))
        return np.nan

    # -------------------------
    # Outputs + stats
    # -------------------------
    X_by_patient = {}
    y_by_patient = {}

    # New output:
    # columns are:
    # [target__abdominal_nec_3h, target__los_3h, adverse_event_3h]
    tag_by_patient = {}

    meta_by_patient = {}
    stats_rows = []

    total_before = 0
    total_after = 0

    total_positive_nec_super = 0
    total_positive_los_super = 0
    total_positive_adverse_super = 0

    # -------------------------
    # Process each patient independently
    # -------------------------
    for pid, df_p in df_all.groupby(id_col, sort=False):
        df_p = df_p.copy()

        # Align timestamps
        if align_timestamps:
            df_p["_frame_time"] = align_to_hop_grid(df_p[timestamp_col])
        else:
            df_p["_frame_time"] = df_p[timestamp_col]

        # ----------------------------------------------------
        # Important change:
        # If multiple rows map to the same _frame_time:
        # - features: first
        # - target PMA: mean
        # - adverse tags: max
        #
        # This prevents losing a positive NEC/LOS tag because of keep="first".
        # ----------------------------------------------------
        agg_dict = {}

        for c in feature_cols:
            agg_dict[c] = "first"

        agg_dict[target_col] = "mean"
        agg_dict[nec_col] = "max"
        agg_dict[los_col] = "max"
        agg_dict["target__adverse_event"] = "max"

        df_p = (
            df_p
            .groupby("_frame_time", as_index=True)
            .agg(agg_dict)
            .sort_index()
        )

        # -------------------------
        # Extract patient-level static values ONCE
        # -------------------------
        static_vals = {}

        for c in static_cols:
            if c in df_p.columns:
                non_na = df_p[c].dropna()

                if len(non_na) == 0:
                    static_vals[c] = np.nan
                else:
                    uniq = non_na.unique()

                    if verbose_static_warnings and len(uniq) > 1:
                        print(
                            f"[PID {pid}] WARNING: static col '{c}' has multiple values: "
                            f"{uniq[:10]} (n={len(uniq)})"
                        )

                    static_vals[c] = (
                        float(uniq[0])
                        if np.issubdtype(non_na.dtype, np.number)
                        else uniq[0]
                    )
            else:
                static_vals[c] = np.nan

        if verbose_static_warnings:
            missing_static = [
                c for c, v in static_vals.items()
                if isinstance(v, float) and not np.isfinite(v)
            ]

            if len(missing_static) > 0:
                print(f"[PID {pid}] WARNING: missing static values for: {missing_static}")

        # Features + targets + tags
        feat_df = df_p[feature_cols]
        target_s = df_p[target_col]

        tag_df = df_p[[nec_col, los_col, "target__adverse_event"]].copy()

        if feat_df.shape[0] == 0:
            X_by_patient[pid] = np.empty((0, frames_per_super, len(feature_cols)), dtype=float)
            y_by_patient[pid] = np.empty((0,), dtype=float)
            tag_by_patient[pid] = np.empty((0, 3), dtype=int)

            meta_by_patient[pid] = pd.DataFrame(
                columns=[
                    "super_id",
                    "start_time",
                    "end_time",
                    f"{target_col}_mean",
                    f"{nec_col}_3h",
                    f"{los_col}_3h",
                    "adverse_event_3h",
                    "adverse_positive_pma_w_in_super",
                    "adverse_negative_pma_w_in_super",
                ] + static_cols
            )

            stats_rows.append({
                "patient_id": pid,
                "n_super_before": 0,
                "n_super_after": 0,
                "n_nec_positive_super": 0,
                "n_los_positive_super": 0,
                "n_adverse_positive_super": 0,
                "retention_ratio": np.nan,
            })

            continue

        t_min = feat_df.index.min()
        t_max = feat_df.index.max()

        # Candidate superwindow starts
        starts = []
        cur = t_min

        while cur <= t_max:
            starts.append(cur)
            cur = cur + super_stride

        n_super_before = len(starts)
        total_before += n_super_before

        blocks = []
        y_list = []
        tag_list = []
        meta_rows = []

        # Define QC columns
        if qc_ignore_static:
            qc_cols = [c for c in feature_cols if c not in static_cols]

            if len(qc_cols) == 0:
                qc_cols = list(feature_cols)
        else:
            qc_cols = list(feature_cols)

        for s_time in starts:
            grid = pd.date_range(start=s_time, periods=frames_per_super, freq=hop)

            # Reindex to fixed grid
            block_df = feat_df.reindex(grid)

            # Fill static columns as constants across the grid
            for c in static_cols:
                if c in block_df.columns:
                    block_df[c] = static_vals[c]

            # QC: per-feature valid count
            per_feature_valid_counts = block_df[qc_cols].notna().sum(axis=0)

            if (per_feature_valid_counts < min_valid_frames_per_feature).any():
                continue

            # PMA target mean over the same grid
            target_block = target_s.reindex(grid).to_numpy(dtype=float)
            target_mean = safe_nanmean(target_block)

            # Tag block over the same grid
            tag_block = tag_df.reindex(grid)

            # For labels:
            # missing tag values are treated as 0 for max aggregation
            tag_block_for_label = tag_block.fillna(0).astype(int)

            nec_3h = int(tag_block_for_label[nec_col].max())
            los_3h = int(tag_block_for_label[los_col].max())
            adverse_3h = int(tag_block_for_label["target__adverse_event"].max())

            # Keep PMA values where adverse tag is positive / negative inside this 3h superwindow
            observed_adverse_mask = tag_block["target__adverse_event"].notna()
            positive_mask = observed_adverse_mask & (tag_block_for_label["target__adverse_event"] == 1)
            negative_mask = observed_adverse_mask & (tag_block_for_label["target__adverse_event"] == 0)

            adverse_positive_pma = pma_list_to_str(target_block[positive_mask.to_numpy()])
            adverse_negative_pma = pma_list_to_str(target_block[negative_mask.to_numpy()])

            blocks.append(block_df.to_numpy(dtype=float))
            y_list.append(target_mean)
            tag_list.append([nec_3h, los_3h, adverse_3h])

            meta_row = {
                "super_id": len(blocks) - 1,
                "start_time": s_time,
                "end_time": s_time + super_len,
                f"{target_col}_mean": target_mean,

                # 3h labels
                f"{nec_col}_3h": nec_3h,
                f"{los_col}_3h": los_3h,
                "adverse_event_3h": adverse_3h,

                # PMA records inside this 3h superwindow
                "adverse_positive_pma_w_in_super": adverse_positive_pma,
                "adverse_negative_pma_w_in_super": adverse_negative_pma,
            }

            for c in static_cols:
                meta_row[c] = static_vals.get(c, np.nan)

            meta_rows.append(meta_row)

        n_super_after = len(blocks)
        total_after += n_super_after

        if n_super_after > 0:
            X_super = np.stack(blocks, axis=0)
            y_super = np.array(y_list, dtype=float)
            tag_super = np.array(tag_list, dtype=int)
        else:
            X_super = np.empty((0, frames_per_super, len(feature_cols)), dtype=float)
            y_super = np.empty((0,), dtype=float)
            tag_super = np.empty((0, 3), dtype=int)

        X_by_patient[pid] = X_super
        y_by_patient[pid] = y_super
        tag_by_patient[pid] = tag_super
        meta_by_patient[pid] = pd.DataFrame(meta_rows)

        if n_super_after > 0:
            n_nec_positive_super = int(tag_super[:, 0].sum())
            n_los_positive_super = int(tag_super[:, 1].sum())
            n_adverse_positive_super = int(tag_super[:, 2].sum())
        else:
            n_nec_positive_super = 0
            n_los_positive_super = 0
            n_adverse_positive_super = 0

        total_positive_nec_super += n_nec_positive_super
        total_positive_los_super += n_los_positive_super
        total_positive_adverse_super += n_adverse_positive_super

        stats_rows.append({
            "patient_id": pid,
            "n_super_before": n_super_before,
            "n_super_after": n_super_after,
            "n_nec_positive_super": n_nec_positive_super,
            "n_los_positive_super": n_los_positive_super,
            "n_adverse_positive_super": n_adverse_positive_super,
            "retention_ratio": (n_super_after / n_super_before) if n_super_before > 0 else np.nan,
        })

    stats_df = pd.DataFrame(stats_rows)

    out = {
        "params": {
            "id_col": id_col,
            "timestamp_col": timestamp_col,
            "feature_prefix": feature_prefix,
            "target_col": target_col,
            "nec_col": nec_col,
            "los_col": los_col,
            "adverse_label_col": "target__adverse_event",
            "frame_len_minutes": frame_len_minutes,
            "frame_hop_minutes": frame_hop_minutes,
            "superwindow_minutes": superwindow_minutes,
            "super_stride_minutes": super_stride_minutes,
            "frames_per_super": frames_per_super,
            "min_valid_frames_per_feature": min_valid_frames_per_feature,
            "align_timestamps": align_timestamps,
            "align_method": align_method,
            "static_cols": list(static_cols),
            "qc_ignore_static": qc_ignore_static,
        },

        "feature_cols": feature_cols,

        # Model input
        "X_by_patient": X_by_patient,

        # PMA target
        "y_by_patient": y_by_patient,

        # New: 3h adverse labels
        # tag columns:
        # 0 = target__abdominal_nec_3h
        # 1 = target__los_3h
        # 2 = adverse_event_3h
        "tag_by_patient": tag_by_patient,
        "tag_cols": [
            f"{nec_col}_3h",
            f"{los_col}_3h",
            "adverse_event_3h",
        ],

        # Metadata, contains start/end time and labels for each superwindow
        "meta_by_patient": meta_by_patient,

        "stats_per_patient": stats_df,

        "stats_global": {
            "total_super_before": int(total_before),
            "total_super_after": int(total_after),
            "global_retention_ratio": (total_after / total_before) if total_before > 0 else np.nan,

            "total_nec_positive_super": int(total_positive_nec_super),
            "total_los_positive_super": int(total_positive_los_super),
            "total_adverse_positive_super": int(total_positive_adverse_super),
        }
    }

    with open(output_pkl_path, "wb") as f:
        pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("Saved:", output_pkl_path)
    print("Total superwindows before QC:", total_before)
    print("Total superwindows after QC :", total_after)
    print("Positive NEC 3h windows     :", total_positive_nec_super)
    print("Positive LOS 3h windows     :", total_positive_los_super)
    print("Positive adverse 3h windows :", total_positive_adverse_super)

    return out

In [ ]:
out = superwindow_crop_pkl_timebased(
    input_pkl_path="/mnt/home/qinqiu/thesis/data/main_data__keep_all_rows_for_nec_or_los_patients__with_nec_los_tags.pkl",
    output_pkl_path="/mnt/home/qinqiu/thesis/data/data_patients_superwindow_3h_stride60m_timebased_featsQC__nec_los_allrows.pkl",
    id_col="ids__uid",
    timestamp_col="timestamp",
    superwindow_minutes=180,
    super_stride_minutes=60,
    min_valid_frames_per_feature=28,
    target_col="target__pma_w",
    feature_prefix="feats__",
)

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from typing import Dict, Any


# =========================================================
# Paths
# =========================================================
IN_PKL = (
    "/mnt/home/qinqiu/thesis/data/"
    "data_patients_superwindow_3h_stride60m_timebased_featsQC__nec_los_allrows.pkl"
)

OUT_PKL = (
    "/mnt/home/qinqiu/thesis/data/"
    "data_patients_superwindow_3h_stride60m_timebased_featsQC__nec_los_allrows_imputed_2ch_CTHW.pkl"
)

DROP_LENGTH_MISMATCH = True
REQUIRE_TAGS = True


# =========================================================
# Helper functions
# =========================================================
def safe_n_windows(X):
    if X is None:
        return 0
    try:
        return int(len(X))
    except Exception:
        return 0


def safe_n_labels(y):
    if y is None:
        return 0
    try:
        return int(len(y))
    except Exception:
        return 0


def get_shape(x):
    if x is None:
        return None
    try:
        return tuple(np.shape(x))
    except Exception:
        return None


def get_invalid_reason(X, y, tag=None, meta=None):
    """
    Return None if patient is valid.
    Otherwise return reason for removal.
    """
    if X is None:
        return "X is None"

    if y is None:
        return "y is None"

    if safe_n_windows(X) == 0:
        return "X has zero windows"

    if safe_n_labels(y) == 0:
        return "y has zero labels"

    if not hasattr(X, "ndim"):
        return "X has no ndim attribute"

    if X.ndim != 3:
        return f"X ndim is not 3: X.ndim={X.ndim}, X.shape={X.shape}"

    if DROP_LENGTH_MISMATCH and len(X) != len(y):
        return f"X/y length mismatch: len(X)={len(X)}, len(y)={len(y)}"

    if REQUIRE_TAGS:
        if tag is None:
            return "tag is missing"

        if safe_n_windows(tag) == 0:
            return "tag has zero windows"

        if DROP_LENGTH_MISMATCH and len(tag) != len(X):
            return f"X/tag length mismatch: len(X)={len(X)}, len(tag)={len(tag)}"

    if meta is not None:
        try:
            if DROP_LENGTH_MISMATCH and len(meta) != len(X):
                return f"X/meta length mismatch: len(X)={len(X)}, len(meta)={len(meta)}"
        except Exception:
            return "meta length cannot be checked"

    return None


def _impute_linear_time_with_mask(x_tf: np.ndarray):
    """
    x_tf: (T, F) float array with NaNs

    Returns:
      x_imputed_tf: (T, F) float32, no NaNs
      mask_tf:      (T, F) float32, 1 where original was NaN, else 0

    Imputation:
      per-feature linear interpolation over time.
      Edge NaNs are filled with nearest valid value by np.interp.
      If one feature is entirely NaN, it is filled with 0.
    """
    if x_tf.ndim != 2:
        raise ValueError(f"Expected (T,F), got {x_tf.shape}")

    T, F = x_tf.shape

    mask_tf = np.isnan(x_tf).astype(np.float32)
    x_imputed = x_tf.astype(np.float32, copy=True)

    t_idx = np.arange(T, dtype=np.float32)

    for f in range(F):
        col = x_imputed[:, f]
        nan = np.isnan(col)

        if not nan.any():
            continue

        valid = ~nan

        if valid.sum() == 0:
            col[:] = 0.0
            x_imputed[:, f] = col
            continue

        col[nan] = np.interp(
            t_idx[nan],
            t_idx[valid],
            col[valid]
        ).astype(np.float32)

        if np.isnan(col).any():
            m = np.nanmean(col)
            col[np.isnan(col)] = 0.0 if not np.isfinite(m) else np.float32(m)

        x_imputed[:, f] = col

    return x_imputed, mask_tf


# =========================================================
# Main conversion function
# =========================================================
def convert_adverse_pkl_to_imputed_2ch_cthw_cleaned(
    in_pkl: str,
    out_pkl: str
) -> Dict[str, Any]:

    with open(in_pkl, "rb") as f:
        out = pickle.load(f)

    X_by_patient = out["X_by_patient"]
    y_by_patient = out["y_by_patient"]

    tag_by_patient = out.get("tag_by_patient", {})
    meta_by_patient = out.get("meta_by_patient", {})
    feature_cols = out.get("feature_cols", None)

    original_X_patient_count = len(X_by_patient)
    original_y_patient_count = len(y_by_patient)
    original_tag_patient_count = len(tag_by_patient)
    original_meta_patient_count = len(meta_by_patient)

    x_pids = set(X_by_patient.keys())
    y_pids = set(y_by_patient.keys())

    if REQUIRE_TAGS:
        tag_pids = set(tag_by_patient.keys())
        candidate_pids = sorted(x_pids.intersection(y_pids).intersection(tag_pids))
    else:
        candidate_pids = sorted(x_pids.intersection(y_pids))

    x_only_pids = sorted(x_pids - y_pids)
    y_only_pids = sorted(y_pids - x_pids)

    original_candidate_patient_count = len(candidate_pids)

    original_candidate_X_windows = 0
    original_candidate_y_labels = 0
    original_candidate_tag_windows = 0

    for pid in candidate_pids:
        original_candidate_X_windows += safe_n_windows(X_by_patient.get(pid, None))
        original_candidate_y_labels += safe_n_labels(y_by_patient.get(pid, None))
        original_candidate_tag_windows += safe_n_windows(tag_by_patient.get(pid, None))

    X2_by_patient = {}
    X_by_patient_clean = {}
    y_by_patient_clean = {}
    tag_by_patient_clean = {}
    meta_by_patient_clean = {}

    removed_rows = []

    stats = {
        "total_windows_imputed": 0,
        "total_nan_before": 0,
        "total_nan_after": 0,
    }

    for pid in candidate_pids:
        X = X_by_patient.get(pid, None)
        y = y_by_patient.get(pid, None)
        tag = tag_by_patient.get(pid, None)
        meta = meta_by_patient.get(pid, None)

        reason = get_invalid_reason(X, y, tag=tag, meta=meta)

        if reason is not None:
            removed_rows.append({
                "pid": pid,
                "reason": reason,
                "X_shape": get_shape(X),
                "y_shape": get_shape(y),
                "tag_shape": get_shape(tag),
                "meta_shape": get_shape(meta),
                "n_X_windows": safe_n_windows(X),
                "n_y_labels": safe_n_labels(y),
                "n_tag_windows": safe_n_windows(tag),
            })
            continue

        N, T, F = X.shape

        X2 = np.empty((N, 2, T, F), dtype=np.float32)

        nan_before = int(np.isnan(X).sum())
        stats["total_windows_imputed"] += int(N)
        stats["total_nan_before"] += nan_before

        for i in range(N):
            x_tf = X[i]

            x_imp_tf, mask_tf = _impute_linear_time_with_mask(x_tf)

            # Channel 0 = imputed feature values
            # Channel 1 = missingness mask, 1 means original NaN
            X2[i, 0, :, :] = x_imp_tf
            X2[i, 1, :, :] = mask_tf

        nan_after = int(np.isnan(X2[:, 0, :, :]).sum())
        stats["total_nan_after"] += nan_after

        X2_by_patient[pid] = X2
        X_by_patient_clean[pid] = X
        y_by_patient_clean[pid] = np.asarray(y)

        if tag is not None:
            tag_by_patient_clean[pid] = np.asarray(tag)

        if meta is not None:
            meta_by_patient_clean[pid] = meta.copy()

    removed_df = pd.DataFrame(removed_rows)

    if len(removed_df) > 0:
        removed_df = removed_df.sort_values("pid").reset_index(drop=True)
        removed_patient_count = int(len(removed_df))
        removed_X_windows = int(removed_df["n_X_windows"].sum())
        removed_y_labels = int(removed_df["n_y_labels"].sum())
        removed_tag_windows = int(removed_df["n_tag_windows"].sum())
    else:
        removed_patient_count = 0
        removed_X_windows = 0
        removed_y_labels = 0
        removed_tag_windows = 0

    final_patient_count = int(len(X2_by_patient))
    final_X_windows = int(sum(len(X) for X in X_by_patient_clean.values()))
    final_y_labels = int(sum(len(y) for y in y_by_patient_clean.values()))
    final_tag_windows = int(sum(len(t) for t in tag_by_patient_clean.values()))

    out2 = dict(out)

    out2["X_by_patient"] = X_by_patient_clean
    out2["y_by_patient"] = y_by_patient_clean
    out2["X2_by_patient"] = X2_by_patient

    out2["tag_by_patient"] = tag_by_patient_clean
    out2["meta_by_patient"] = meta_by_patient_clean
    out2["feature_cols"] = feature_cols

    out2["imputation"] = {
        "method": "per-feature linear interpolation over time + missingness mask channel",
        "output_format": "N,C,H,W with C=2; channel 0 = imputed values, channel 1 = missingness mask",
        "channel_0": "imputed feature values",
        "channel_1": "missingness mask; 1 = original NaN, 0 = observed",
        "invalid_patients_removed": True,
        "drop_length_mismatch": DROP_LENGTH_MISMATCH,
        "tags_preserved": True,
        "meta_preserved": True,
    }

    out2["imputation_stats"] = stats

    out2["patient_window_filtering_summary"] = {
        "original_X_by_patient_count": int(original_X_patient_count),
        "original_y_by_patient_count": int(original_y_patient_count),
        "original_tag_by_patient_count": int(original_tag_patient_count),
        "original_meta_by_patient_count": int(original_meta_patient_count),

        "original_candidate_patient_count": int(original_candidate_patient_count),

        "x_only_patient_count": int(len(x_only_pids)),
        "y_only_patient_count": int(len(y_only_pids)),

        "original_candidate_X_windows": int(original_candidate_X_windows),
        "original_candidate_y_labels": int(original_candidate_y_labels),
        "original_candidate_tag_windows": int(original_candidate_tag_windows),

        "removed_patient_count": int(removed_patient_count),
        "removed_X_windows": int(removed_X_windows),
        "removed_y_labels": int(removed_y_labels),
        "removed_tag_windows": int(removed_tag_windows),

        "final_patient_count": int(final_patient_count),
        "final_X_windows": int(final_X_windows),
        "final_y_labels": int(final_y_labels),
        "final_tag_windows": int(final_tag_windows),
    }

    out2["removed_patients"] = removed_rows
    out2["x_only_pids"] = x_only_pids
    out2["y_only_pids"] = y_only_pids

    with open(out_pkl, "wb") as f:
        pickle.dump(out2, f, protocol=pickle.HIGHEST_PROTOCOL)

    out_dir = os.path.dirname(out_pkl)
    out_base = os.path.splitext(os.path.basename(out_pkl))[0]

    removed_csv = os.path.join(out_dir, f"{out_base}_removed_patients.csv")
    valid_csv = os.path.join(out_dir, f"{out_base}_valid_patients.csv")
    summary_txt = os.path.join(out_dir, f"{out_base}_filtering_summary.txt")

    removed_df.to_csv(removed_csv, index=False)

    valid_rows = []

    for pid in sorted(X2_by_patient.keys()):
        X2 = X2_by_patient[pid]
        y = y_by_patient_clean[pid]
        tag = tag_by_patient_clean.get(pid, None)

        row = {
            "pid": pid,
            "X2_shape": tuple(X2.shape),
            "n_windows": len(X2),
            "n_y_labels": len(y),
            "target_mean": float(np.mean(y)),
            "target_median": float(np.median(y)),
        }

        if tag is not None and tag.shape[1] >= 3:
            row["n_nec_positive_windows"] = int(tag[:, 0].sum())
            row["n_los_positive_windows"] = int(tag[:, 1].sum())
            row["n_adverse_positive_windows"] = int(tag[:, 2].sum())

        valid_rows.append(row)

    valid_df = pd.DataFrame(valid_rows).sort_values("pid").reset_index(drop=True)
    valid_df.to_csv(valid_csv, index=False)

    print("\n" + "=" * 90)
    print("Adverse pkl imputation + 2-channel conversion summary")
    print("=" * 90)

    print("\nPatient counts:")
    print(f"Original X_by_patient count:             {original_X_patient_count}")
    print(f"Original y_by_patient count:             {original_y_patient_count}")
    print(f"Original tag_by_patient count:           {original_tag_patient_count}")
    print(f"Original meta_by_patient count:          {original_meta_patient_count}")
    print(f"Original candidate patients:             {original_candidate_patient_count}")
    print(f"Removed patients:                        {removed_patient_count}")
    print(f"Final valid patients:                    {final_patient_count}")

    print("\nWindow / label / tag counts:")
    print(f"Original candidate X windows:            {original_candidate_X_windows}")
    print(f"Original candidate y labels:             {original_candidate_y_labels}")
    print(f"Original candidate tag windows:          {original_candidate_tag_windows}")
    print(f"Removed X windows:                       {removed_X_windows}")
    print(f"Removed y labels:                        {removed_y_labels}")
    print(f"Removed tag windows:                     {removed_tag_windows}")
    print(f"Final X windows:                         {final_X_windows}")
    print(f"Final y labels:                          {final_y_labels}")
    print(f"Final tag windows:                       {final_tag_windows}")

    print("\nImputation stats:")
    print(f"Total windows imputed:                   {stats['total_windows_imputed']}")
    print(f"Total NaNs before imputation:            {stats['total_nan_before']}")
    print(f"Total NaNs after imputation:             {stats['total_nan_after']}")

    if len(valid_df) > 0 and "n_adverse_positive_windows" in valid_df.columns:
        print("\nAdverse tag check:")
        print(f"Patients with adverse-positive windows:  {(valid_df['n_adverse_positive_windows'] > 0).sum()}")
        print(f"Total adverse-positive windows:          {valid_df['n_adverse_positive_windows'].sum()}")

    print("\nSaved files:")
    print(f"Imputed adverse PKL:                     {out_pkl}")
    print(f"Removed patients CSV:                    {removed_csv}")
    print(f"Valid patients CSV:                      {valid_csv}")

    with open(summary_txt, "w") as f:
        f.write("Adverse pkl imputation + 2-channel conversion summary\n")
        f.write("=" * 90 + "\n\n")
        f.write(str(out2["patient_window_filtering_summary"]))
        f.write("\n\n")
        f.write(str(out2["imputation_stats"]))

    print(f"Summary TXT:                             {summary_txt}")

    return out2


# =========================================================
# Run
# =========================================================
adverse_out2 = convert_adverse_pkl_to_imputed_2ch_cthw_cleaned(
    IN_PKL,
    OUT_PKL,
)

print("\n" + "=" * 90)
print("Sanity check")
print("=" * 90)

if len(adverse_out2["X2_by_patient"]) > 0:
    some_pid = next(iter(adverse_out2["X2_by_patient"].keys()))
    x = adverse_out2["X2_by_patient"][some_pid]

    print("Example patient:", some_pid)
    print("X2 shape:", x.shape)
    print("NaN in value channel:", np.isnan(x[:, 0]).any())
    print("NaN in mask channel:", np.isnan(x[:, 1]).any())

    if "tag_by_patient" in adverse_out2:
        tag = adverse_out2["tag_by_patient"][some_pid]
        print("Tag shape:", tag.shape)
        print("Positive NEC windows:", int(tag[:, 0].sum()))
        print("Positive LOS windows:", int(tag[:, 1].sum()))
        print("Positive adverse windows:", int(tag[:, 2].sum()))

    if "meta_by_patient" in adverse_out2:
        meta = adverse_out2["meta_by_patient"][some_pid]
        print("Meta shape:", meta.shape)
        print(meta.head())
else:
    print("No valid patients found.")